<a href="https://colab.research.google.com/github/eng20260311/AIFFEL_quest_eng/blob/master/NLP/NLP02/notebookabdc540598_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

%config InlineBackend.figure_format = 'retina'

fontpath = '/usr/share/fonts/truetype/nanum/NanumBarunGothic.ttf'
font = fm.FontProperties(fname=fontpath, size=9)
plt.rc('font', family='NanumBarunGothic')
mpl.font_manager.findfont(font)

print("슝=3")

슝=3


In [ ]:
# ===== 필수 라이브러리 import =====
import numpy as np            # 수치 연산 (positional encoding 계산 등에 사용)
import torch                  # PyTorch 본체
import torch.nn as nn         # 신경망 모듈 (Linear, LayerNorm, Embedding 등)
import torch.nn.functional as F  # 함수형 API (softmax 등)
import torch.optim as optim   # 옵티마이저 (Adam 등)
import matplotlib.pyplot as plt  # Attention 시각화용 플롯

import re                     # 정규표현식 (텍스트 전처리)
import os                     # OS 관련 유틸
import io                     # 입출력 스트림
import time                   # 학습 시간 측정
import random                 # 배치 셔플
import math                   # 수학 함수

import seaborn  # Attention 시각화 시 히트맵을 그리기 위해 필요!

# 현재 사용 중인 PyTorch 버전 출력
print(torch.__version__)


2.10.0+cu128


In [ ]:
# ===== 한국어-영어 병렬 코퍼스 다운로드 =====
# jungyeul의 GitHub 저장소에서 한영 뉴스 병렬 코퍼스(train)를 받아옵니다.
import os

# 1) GitHub에서 tar.gz 압축 파일 다운로드 (~/content 디렉토리에 저장)
#!wget https://github.com/jungyeul/korean-parallel-corpora/raw/master/korean-english-news-v1/korean-english-park.train.tar.gz -P ~/content

# 2) .gz 해제 → .tar 파일이 됨
#!gzip -d ~/content/korean-english-park.train.tar.gz

# 3) .tar 해제 → 한국어/영어 텍스트 파일 두 개가 추출됨
#!tar -xvf ~/content/korean-english-park.train.tar

# 4) 추출된 파일 경로 지정 (이후 단계에서 읽어들임)
kor_path = "/kaggle/input/datasets/hyunheekim78/kor-eng/korean-english-park.train.ko"  # 한국어 원문
eng_path = "/kaggle/input/datasets/hyunheekim78/kor-eng/korean-english-park.train.en"  # 영어 번역문


In [ ]:
# ===== 데이터 정제 함수 =====
def clean_corpus(kor_path, eng_path):
    """한국어-영어 병렬 텍스트 파일을 읽어 중복을 제거한 페어 리스트를 반환합니다."""
    # 한국어 파일을 줄 단위로 읽기
    with open(kor_path, "r") as f:
        kor = f.read().splitlines()
    # 영어 파일을 줄 단위로 읽기
    with open(eng_path, "r") as f:
        eng = f.read().splitlines()

    # 두 코퍼스의 줄 수가 같아야 (병렬이어야) 함
    assert len(kor) == len(eng)

    # 한국어\t영어 형식으로 합친 뒤 set으로 중복 제거 → 다시 리스트로
    cleaned_corpus = list(set(["\t".join([k, e]) for k, e in zip(kor, eng)]))

    return cleaned_corpus

# 정제된 (한국어, 영어) 페어 코퍼스 생성
cleaned_corpus = clean_corpus(kor_path, eng_path)


In [ ]:
# ===== 문장 단위 전처리 함수 =====
def preprocess_sentence(sentence):
    """하나의 문장에 대해 소문자 변환·특수문자 제거·구두점 분리 등을 수행합니다."""

    # 1) 영문 소문자화
    sentence = sentence.lower()

    # 2) 알파벳·한글·기본 구두점(?.!,)을 제외한 모든 문자를 공백으로 치환
    sentence = re.sub(r"[^a-zA-Z가-힣?.!,]+", " ", sentence)

    # 3) 구두점 앞뒤에 공백을 넣어 토큰화 시 분리되도록 함
    sentence = re.sub(r"([?.!,])", r" \1 ", sentence)

    # 4) 연속된 공백을 하나로 압축
    sentence = re.sub(r'[" "]+', " ", sentence)

    return sentence


In [ ]:
# ===== SentencePiece 토크나이저 학습 =====
import re

def generate_tokenizer(corpus,
                       vocab_size,
                       lang="ko",
                       pad_id=0,   # <pad> 토큰 ID
                       bos_id=1,   # <bos> 토큰 ID (문장 시작)
                       eos_id=2,   # <eos> 토큰 ID (문장 끝)
                       unk_id=3):  # <unk> 토큰 ID (미등록 단어)
    """코퍼스를 받아 SentencePiece 모델을 학습하고 토크나이저 객체를 반환합니다."""
    file = "./%s_corpus.txt" % lang   # 임시 코퍼스 파일 경로
    model = "%s_spm" % lang           # 학습된 모델의 접두어

    # 1) 코퍼스를 한 줄씩 텍스트 파일로 저장 (SentencePiece 학습 입력)
    with open(file, 'w') as f:
        for row in corpus:
            f.write(str(row) + '\n')

    # 2) SentencePiece 학습 실행 (vocab_size 만큼의 subword 사전 구축)
    import sentencepiece as spm
    spm.SentencePieceTrainer.Train(
        '--input=./%s --model_prefix=%s --vocab_size=%d '
        % (file, model, vocab_size) +
        '--pad_id=%d --bos_id=%d --eos_id=%d --unk_id=%d'  # 수정: pad_id== → pad_id=, 그리고 앞 문자열 끝에 공백 추가
        % (pad_id, bos_id, eos_id, unk_id)
    )

    # 3) 학습된 모델 로드
    tokenizer = spm.SentencePieceProcessor()
    tokenizer.Load('%s.model' % model)

    return tokenizer


# 소스(한국어)/타깃(영어) 단어 사전 크기
SRC_VOCAB_SIZE = TGT_VOCAB_SIZE = 20000

# 전처리된 문장들을 담을 리스트
eng_corpus = []
kor_corpus = []

# 정제된 페어를 분리하고 각각 전처리
for pair in cleaned_corpus:
    k, e = pair.split("\t")
    kor_corpus.append(preprocess_sentence(k))
    eng_corpus.append(preprocess_sentence(e))

# 한국어/영어 각각에 대해 SentencePiece 토크나이저 학습
ko_tokenizer = generate_tokenizer(kor_corpus, SRC_VOCAB_SIZE, "ko")
en_tokenizer = generate_tokenizer(eng_corpus, TGT_VOCAB_SIZE, "en")

# 디코더 입력에는 항상 <bos>, <eos>를 붙이도록 설정
en_tokenizer.set_encode_extra_options("bos:eos")


sentencepiece_trainer.cc(178) LOG(INFO) Running command: --input=././ko_corpus.txt --model_prefix=ko_spm --vocab_size=20000 --pad_id=0 --bos_id=1 --eos_id=2 --unk_id=3
sentencepiece_trainer.cc(78) LOG(INFO) Starts training with : 
trainer_spec {
  input: ././ko_corpus.txt
  input_format: 
  model_prefix: ko_spm
  model_type: UNIGRAM
  vocab_size: 20000
  self_test_sample_size: 0
  character_coverage: 0.9995
  input_sentence_size: 0
  shuffle_input_sentence: 1
  seed_sentencepiece_size: 1000000
  shrinking_factor: 0.75
  max_sentence_length: 4192
  num_threads: 16
  num_sub_iterations: 2
  max_sentencepiece_length: 16
  split_by_unicode_script: 1
  split_by_number: 1
  split_by_whitespace: 1
  split_digits: 0
  pretokenization_delimiter: 
  treat_whitespace_as_suffix: 0
  allow_whitespace_only_pieces: 0
  required_chars: 
  byte_fallback: 0
  vocabulary_output_piece_score: 1
  train_extremely_large_corpus: 0
  seed_sentencepieces_file: 
  hard_vocab_limit: 1
  use_all_vocab: 0
  unk_id:

True

In [ ]:
# ===== 토큰화 + 길이 필터링 + 패딩 =====
import torch
import torch.nn.functional as F
from tqdm.notebook import tqdm  # 진행 과정 시각화

src_corpus = []   # 소스(한국어) 토큰 시퀀스
tgt_corpus = []   # 타깃(영어) 토큰 시퀀스

# 두 코퍼스의 길이가 같은지 확인
assert len(kor_corpus) == len(eng_corpus)

# 토큰의 길이가 50 이하인 문장만 남깁니다.
# (긴 문장은 메모리·연산량 부담이 크므로 제외)
for idx in tqdm(range(len(kor_corpus))):
    src_tokens = ko_tokenizer.encode_as_ids(kor_corpus[idx])  # 한국어 → ID 시퀀스
    tgt_tokens = en_tokenizer.encode_as_ids(eng_corpus[idx])  # 영어 → ID 시퀀스

    # 양쪽 모두 50 토큰 이하인 페어만 사용
    if len(src_tokens) <= 50 and len(tgt_tokens) <= 50:
        src_corpus.append(torch.tensor(src_tokens, dtype=torch.long))
        tgt_corpus.append(torch.tensor(tgt_tokens, dtype=torch.long))


def pad_sequences(sequences, padding_value=0):
    """가변 길이 시퀀스들을 배치 내 가장 긴 길이에 맞춰 0으로 패딩합니다."""
    return torch.nn.utils.rnn.pad_sequence(sequences, batch_first=True, padding_value=padding_value)

# 패딩처리를 완료하여 학습용 데이터를 완성합니다.
enc_train = pad_sequences(src_corpus, padding_value=0)  # 인코더 입력
dec_train = pad_sequences(tgt_corpus, padding_value=0)  # 디코더 입력

# (문장 수, 최대 시퀀스 길이) 형태 확인
print(enc_train.shape, dec_train.shape)


  0%|          | 0/78968 [00:00<?, ?it/s]

torch.Size([72107, 50]) torch.Size([72107, 50])


In [ ]:
# ===== Positional Encoding =====
#  위치 정보를 sin/cos 함수로 인코딩해서 더해줍니다.

def positional_encoding(pos, d_model):
    """길이 pos × 차원 d_model 크기의 positional encoding 테이블을 반환합니다."""

    def cal_angle(position, i):
        # 논문 공식: position / 10000^(2i/d_model)
        return position / np.power(10000, int(i) / d_model)

    def get_posi_angle_vec(position):
        # 한 position에 대해 d_model 차원의 angle 벡터 생성
        return [cal_angle(position, i) for i in range(d_model)]

    # (pos, d_model) 모양의 각도 테이블
    sinusoid_table = np.array([get_posi_angle_vec(pos_i) for pos_i in range(pos)])

    # 짝수 인덱스 차원에는 sin, 홀수 인덱스 차원에는 cos 적용
    sinusoid_table[:, 0::2] = np.sin(sinusoid_table[:, 0::2])
    sinusoid_table[:, 1::2] = np.cos(sinusoid_table[:, 1::2])

    return sinusoid_table


In [ ]:
# ===== Multi-Head Attention =====
import torch
import torch.nn as nn

class MultiHeadAttention(nn.Module):
    """Transformer의 핵심 모듈: Query/Key/Value를 여러 헤드로 나눠 attention 계산."""

    def __init__(self, d_model, num_heads):
        super(MultiHeadAttention, self).__init__()
        self.num_heads = num_heads
        self.d_model = d_model
        self.depth = d_model // num_heads       #각 헤드의 차원

        # d_model이 num_heads로 나누어 떨어져야 함
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"

        # Q, K, V를 만드는 선형 변환
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)

        # 헤드들을 합친 뒤 한 번 더 선형 변환
        self.linear = nn.Linear(d_model, d_model)

    def scaled_dot_product_attention(self, Q, K, V, mask=None):
        """Scaled Dot-Product Attention: softmax(QK^T / sqrt(d_k)) * V"""
        d_k = K.shape[-1]

        # Q와 K의 내적 → 어떤 위치를 얼마나 볼지에 대한 점수
        QK = torch.matmul(Q, K.transpose(-2, -1))

        # sqrt(d_k)로 스케일링하여 gradient가 너무 작아지지 않게 함
        scaled_qk = QK / torch.sqrt(torch.tensor(d_k, dtype=torch.float32))

        # 마스크가 있는 위치(예: 패딩, 미래 토큰)는 -inf로 만들어 softmax에서 0이 되게 함
        if mask is not None:
            scaled_qk = scaled_qk.masked_fill(mask == 0, float('-1e9'))

        # softmax로 attention 가중치 정규화
        attentions = F.softmax(scaled_qk, dim=-1)

        # 가중치를 V에 곱해 최종 출력 계산
        out = torch.matmul(attentions, V)

        return out, attentions

    def split_heads(self, x):
        """(batch, seq, d_model) → (batch, num_heads, seq, depth) 로 분할"""
        bsz, seq_len, d_model = x.shape
        x = x.view(bsz, seq_len, self.num_heads, self.depth)
        x = x.permute(0, 2, 1, 3)  # (batch, num_heads, seq, depth)
        return x

    def combine_heads(self, x):
        """(batch, num_heads, seq, depth) → (batch, seq, d_model) 으로 다시 합침"""
        bsz, num_heads, seq_len, depth = x.shape
        x = x.permute(0, 2, 1, 3).contiguous()
        x = x.view(bsz, seq_len, self.d_model)
        return x

    def forward(self, Q, K, V, mask=None):
        # 1) Q, K, V 선형 변환
        WQ = self.W_q(Q)
        WK = self.W_k(K)
        WV = self.W_v(V)

        # 2) 헤드 분할
        WQ_splits = self.split_heads(WQ)
        WK_splits = self.split_heads(WK)
        WV_splits = self.split_heads(WV)

        # 3) Scaled Dot-Product Attention 적용
        out, attention_weights = self.scaled_dot_product_attention(
            WQ_splits, WK_splits, WV_splits, mask)

        # 4) 헤드들 다시 합치고
        out = self.combine_heads(out)

        # 5) 마지막 선형 변환
        out = self.linear(out)

        return out, attention_weights


In [ ]:
# ===== Position-wise Feed Forward Network =====
# 각 위치(토큰)별로 동일하게 적용되는 2층 MLP. 채널 차원을 확장했다 다시 줄임.

class PoswiseFeedForwardNet(nn.Module):
    def __init__(self, d_model, d_ff):
        super(PoswiseFeedForwardNet, self).__init__()
        self.fc1 = nn.Linear(d_model, d_ff)  # d_model → d_ff (확장, 보통 4배)
        self.fc2 = nn.Linear(d_ff, d_model)  # d_ff → d_model (축소)
        self.relu = nn.ReLU()                # 비선형 활성화

    def forward(self, x):
        out = self.fc1(x)    # 차원 확장
        out = self.relu(out) # 비선형성 부여
        out = self.fc2(out)  # 다시 원래 차원으로

        return out


In [ ]:
# ===== Encoder Layer (Transformer 인코더의 한 층) =====
# 구성: Self-Attention → Add&Norm → FFN → Add&Norm

class EncoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout):
        super(EncoderLayer, self).__init__()

        # 1) Self-Attention 모듈
        self.enc_self_attn = MultiHeadAttention(d_model, n_heads)
        # 2) Position-wise FFN
        self.ffn = PoswiseFeedForwardNet(d_model, d_ff)

        # Layer Normalization 두 개 (각 sublayer 앞에서 적용 — Pre-LN 방식)
        self.norm_1 = nn.LayerNorm(d_model, eps=1e-6)
        self.norm_2 = nn.LayerNorm(d_model, eps=1e-6)

        # 드롭아웃 (과적합 방지)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask):
        """
        Multi-Head Attention
        """
        residual = x                                          # 잔차 연결용 저장
        out = self.norm_1(x)                                  # Pre-LN
        out, enc_attn = self.enc_self_attn(out, out, out, mask)  # self-attention (Q=K=V)
        out = self.dropout(out)
        out += residual                                       # residual connection

        """
        Position-Wise Feed Forward Network
        """
        residual = out
        out = self.norm_2(out)
        out = self.ffn(out)
        out = self.dropout(out)
        out += residual

        return out, enc_attn  # 출력과 attention 가중치 반환


In [ ]:
# ===== Decoder Layer (Transformer 디코더의 한 층) =====
# 구성: Masked Self-Attention → Encoder-Decoder Attention → FFN
# (각 sublayer마다 Add&Norm 적용)

class DecoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout):
        super(DecoderLayer, self).__init__()

        # 1) 디코더 self-attention (미래 토큰은 보지 못하도록 causal mask 사용)
        self.dec_self_attn = MultiHeadAttention(d_model, num_heads)
        # 2) 인코더 출력을 참조하는 cross-attention
        self.enc_dec_attn = MultiHeadAttention(d_model, num_heads)

        # 3) Position-wise FFN
        self.ffn = PoswiseFeedForwardNet(d_model, d_ff)

        # 세 개의 sublayer마다 LayerNorm
        self.norm_1 = nn.LayerNorm(d_model, eps=1e-6)
        self.norm_2 = nn.LayerNorm(d_model, eps=1e-6)
        self.norm_3 = nn.LayerNorm(d_model, eps=1e-6)

        self.dropout = nn.Dropout(dropout)

    def forward(self, x, enc_out, causality_mask, padding_mask):
        """
        Masked Multi-Head Attention (디코더 self-attention)
        """
        residual = x
        out = self.norm_1(x)
        # Q=K=V=디코더 입력, 미래 토큰 차단용 마스크 사용
        out, dec_attn = self.dec_self_attn(out, out, out, padding_mask)
        out = self.dropout(out)
        out += residual

        """
        Multi-Head Attention (Encoder-Decoder Cross Attention)
        """
        residual = out
        out = self.norm_2(out)
        # Q=디코더 출력, K=V=인코더 출력 → 소스 문장에 attention
        out, dec_enc_attn = self.enc_dec_attn(out, enc_out, enc_out, causality_mask)
        out = self.dropout(out)
        out += residual

        """
        Position-Wise Feed Forward Network
        """
        residual = out
        out = self.norm_3(out)
        out = self.ffn(out)
        out = self.dropout(out)
        out += residual

        return out, dec_attn, dec_enc_attn  # 디코더 self-attn / cross-attn 가중치 함께 반환


In [ ]:
# ===== Encoder (EncoderLayer를 n_layers개 쌓은 모듈) =====

class Encoder(nn.Module):
    def __init__(self, n_layers, d_model, n_heads, d_ff, dropout):
        super(Encoder, self).__init__()
        self.n_layers = n_layers

        # n_layers개의 EncoderLayer를 ModuleList로 쌓아둠
        self.enc_layers = nn.ModuleList([
            EncoderLayer(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)
        ])

        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask):
        out = x
        enc_attns = []  # 시각화/분석용 attention 가중치 저장

        # 각 레이어를 차례로 통과
        for i in range(self.n_layers):
            out, enc_attn = self.enc_layers[i](out, mask)
            enc_attns.append(enc_attn)

        return out, enc_attns


In [ ]:
# ===== Decoder (DecoderLayer를 n_layers개 쌓은 모듈) =====

class Decoder(nn.Module):
    def __init__(self, n_layers, d_model, n_heads, d_ff, dropout):
        super(Decoder, self).__init__()
        self.n_layers = n_layers

        # n_layers개의 DecoderLayer
        self.dec_layers = nn.ModuleList([
            DecoderLayer(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)
        ])

    def forward(self, x, enc_out, causality_mask, padding_mask):
        out = x

        dec_attns = []      # 디코더 self-attention 가중치
        dec_enc_attns = []  # 인코더-디코더 cross-attention 가중치

        # 각 디코더 레이어를 순차 통과
        for i in range(self.n_layers):
            out, dec_attn, dec_enc_attn = self.dec_layers[i](
                out, enc_out, causality_mask, padding_mask
            )

            dec_attns.append(dec_attn)
            dec_enc_attns.append(dec_enc_attn)

        return out, dec_attns, dec_enc_attns


In [ ]:
# ===== 전체 Transformer 모델 =====
# Encoder + Decoder + Embedding + Positional Encoding + 최종 출력층(fc)

class Transformer(nn.Module):
    def __init__(self,
                 n_layers,       # 인코더/디코더의 레이어 수
                 d_model,        # 임베딩 및 hidden 차원
                 n_heads,        # 멀티헤드 수
                 d_ff,           # FFN 내부 차원
                 src_vocab_size, # 소스 사전 크기
                 tgt_vocab_size, # 타깃 사전 크기
                 pos_len,        # positional encoding 최대 길이
                 dropout=0.2,
                 shared=True):   # 디코더 임베딩과 출력층 가중치를 공유할지

        super(Transformer, self).__init__()

        self.d_model = d_model
        self.shared = shared

        # 소스/타깃 토큰 임베딩
        self.enc_emb = nn.Embedding(src_vocab_size, d_model)
        self.dec_emb = nn.Embedding(tgt_vocab_size, d_model)

        # 미리 계산된 positional encoding 테이블 (학습되지 않음)
        self.pos_encoding = torch.FloatTensor(positional_encoding(pos_len, d_model))
        self.dropout = nn.Dropout(dropout)

        # 인코더와 디코더
        self.encoder = Encoder(n_layers, d_model, n_heads, d_ff, dropout)
        self.decoder = Decoder(n_layers, d_model, n_heads, d_ff, dropout)

        # 최종 logits 생성용 선형층 (vocab 크기로 투영)
        self.fc = nn.Linear(d_model, tgt_vocab_size)

        # weight tying: 디코더 임베딩과 출력층 가중치를 공유 (파라미터 수 감소 + 일반화 ↑)
        if shared:
            self.fc.weight = self.dec_emb.weight

    def embedding(self, emb, x):
        """토큰 임베딩 + positional encoding + dropout 적용"""
        seq_len = x.shape[1]

        out = emb(x)  # 토큰 ID → 임베딩 벡터

        # weight tying 사용 시 임베딩에 sqrt(d_model) 스케일링 (논문 권장)
        if self.shared:
            out *= torch.sqrt(torch.tensor(self.d_model, dtype=torch.float32))

        # 시퀀스 길이만큼 잘라 positional encoding 더하기
        pos_enc = self.pos_encoding[:seq_len, :].unsqueeze(0).to(x.device)
        out += pos_enc

        out = self.dropout(out)
        return out

    def forward(self, enc_in, dec_in, enc_mask, causality_mask, dec_mask):
        # 1) 입력 임베딩 (소스/타깃 각각)
        enc_in = self.embedding(self.enc_emb, enc_in)
        dec_in = self.embedding(self.dec_emb, dec_in)

        # 2) 인코더 통과
        enc_out, enc_attns = self.encoder(enc_in, enc_mask)

        # 3) 디코더 통과 (인코더 출력을 cross-attention에 사용)
        dec_out, dec_attns, dec_enc_attns = self.decoder(dec_in, enc_out, causality_mask, dec_mask)

        # 4) 최종 logits 생성 (각 위치별 vocab 분포)
        logits = self.fc(dec_out)

        # logits과 시각화용 attention 들을 함께 반환
        return logits, enc_attns, dec_attns, dec_enc_attns


In [ ]:
# ===== 마스크 생성 함수들 =====

def generate_padding_mask(seq):
    """ 패딩된 부분(0)을 1로 변환하여 마스크 생성 """
    # seq == 0인 위치(=패딩)는 1, 나머지는 0인 마스크
    mask = (seq == 0).float()
    # attention 계산에 broadcasting 되도록 차원 추가: (batch, 1, 1, seq_len)
    return mask[:, None, None, :]

def generate_causality_mask(src_len, tgt_len):
    """ 미래 정보를 참조하지 않도록 Causal Mask 생성 """
    # 하삼각 행렬: 자기 자신과 과거만 보고 미래는 가림
    mask = 1 - torch.cumsum(torch.eye(src_len, tgt_len), dim=0)
    return mask.float()

def generate_masks(src, tgt):
    """ Encoder-Decoder에서 사용할 마스크 생성 """
    # 인코더용 패딩 마스크 (소스 문장의 패딩 가림)
    enc_mask = generate_padding_mask(src)
    # 디코더 입력용 패딩 마스크
    dec_mask = generate_padding_mask(tgt)

    # 디코더 self-attention용 causal mask + padding mask 결합
    dec_causality_mask = generate_causality_mask(tgt.shape[1], tgt.shape[1])
    dec_mask = torch.max(dec_mask, dec_causality_mask.to(dec_mask.device))

    # 인코더-디코더 cross-attention용 마스크
    dec_enc_causality_mask = generate_causality_mask(tgt.shape[1], src.shape[1])
    dec_enc_mask = torch.max(enc_mask, dec_enc_causality_mask.to(enc_mask.device))

    return enc_mask, dec_enc_mask, dec_mask


In [ ]:
# ===== Transformer 모델 인스턴스 생성 (테스트용) =====
import numpy as np

transformer = Transformer(
    n_layers=2,                  # 인코더/디코더 레이어 수 (작게 설정해 빠른 테스트)
    d_model=512,                 # 임베딩/hidden 차원
    n_heads=8,                   # 멀티헤드 수
    d_ff=2048,                   # FFN 내부 차원
    src_vocab_size=SRC_VOCAB_SIZE,  # 한국어 사전 크기
    tgt_vocab_size=TGT_VOCAB_SIZE,  # 영어 사전 크기
    pos_len=200,                 # positional encoding 최대 길이
    dropout=0.2,
    shared=True                  # 디코더 임베딩과 출력층 weight tying
)


In [ ]:
# ===== Transformer 논문의 Warmup Learning Rate Scheduler =====
# 학습 초반 warmup_steps 동안은 lr을 선형 증가, 이후 step^-0.5로 감소

import math

class LearningRateScheduler(torch.optim.lr_scheduler._LRScheduler):
    def __init__(self, optimizer, d_model, warmup_steps=4000, last_epoch=-1):
        self.d_model = d_model            # 모델 차원 (스케일링에 사용)
        self.warmup_steps = warmup_steps  # warmup 스텝 수
        super(LearningRateScheduler, self).__init__(optimizer, last_epoch)

    def get_lr(self):
        # 현재 step (0이 되지 않도록 최소 1)
        step = max(1, self.last_epoch)

        # 논문 공식: lr = d_model^-0.5 * min(step^-0.5, step * warmup^-1.5)
        arg1 = step ** -0.5
        arg2 = step * (self.warmup_steps ** -1.5)
        lr = (self.d_model ** -0.5) * min(arg1, arg2)

        return [lr for _ in self.base_lrs]


In [ ]:
# ===== 실제 학습에 사용할 모델 + 옵티마이저 + 스케줄러 =====
# 위에서 만든 transformer는 테스트용. 여기서는 더 큰 6-layer Transformer를 새로 만듦.

model = Transformer(src_vocab_size=SRC_VOCAB_SIZE,
                    tgt_vocab_size=TGT_VOCAB_SIZE,
                    d_model=512,
                    n_heads=8,
                    n_layers=6,          # 인코더/디코더 레이어 수
                    d_ff=2048,           # FFN 내부 차원 (보통 d_model * 4)
                    pos_len=512)         # positional encoding 최대 길이 (시퀀스 최대 길이)

# 논문 권장 Adam 설정 (β2=0.98, eps=1e-9). lr은 scheduler가 조정하므로 초기값 0
optimizer = torch.optim.Adam(model.parameters(), lr=0, betas=(0.9, 0.98), eps=1e-9)

# Warmup → decay 스케줄 적용
learning_rate = LearningRateScheduler(optimizer, d_model=512)


In [ ]:
# ===== 손실 함수 정의 (패딩 토큰 무시) =====

# reduction="none": 위치별 손실을 그대로 받아 마스킹 후 평균
loss_object = torch.nn.CrossEntropyLoss(reduction="none")


def loss_function(real, pred):
    """패딩 토큰(0)을 제외한 위치에서만 cross-entropy 손실을 계산합니다."""

    # 1. 패딩 토큰(0)을 제외하기 위한 마스크 생성 (기존 [64, 49])
    mask = real != 0

    # 2. PyTorch CrossEntropy 차원 기준에 맞게 평탄화(Flatten)
    # pred: [64, 49, 20000] -> [64 * 49, 20000]
    # real: [64, 49] -> [64 * 49]
    # view 대신 reshape을 사용하여 메모리 불연속 문제를 우회합니다.
    pred_flat = pred.reshape(-1, pred.size(-1))
    real_flat = real.reshape(-1)
    mask_flat = mask.reshape(-1).float()


    # 3. 손실 계산 및 마스킹 적용 (패딩 위치 손실은 0이 됨)
    loss_ = loss_object(pred_flat, real_flat)
    loss_ *= mask_flat

    # 4. Masking 되지 않은 입력의 개수로 Scaling하여 반환
    # (실제 유효 토큰 수로 나눠 평균 계산)
    return loss_.sum() / mask_flat.sum()


In [ ]:
# Train Step 함수 정의
def train_step(src, tgt, model, optimizer):
    # 정답(gold)은 디코더 입력에서 첫 토큰(<bos>)을 뺀 나머지
    # → 디코더는 i번째 토큰을 보고 (i+1)번째 토큰을 예측
    gold = tgt[:, 1:]

    # 학습에 필요한 마스크들 생성
    enc_mask, dec_enc_mask, dec_mask = generate_masks(src, tgt)

    # 계산된 loss에 대해 역전파(Backpropagation)를 적용해 학습을 진행합니다.
    optimizer.zero_grad()  # 이전 step의 gradient 초기화

    # 모델 forward (예측값 + attention 가중치들 반환)
    predictions, enc_attns, dec_attns, dec_enc_attns = model(src, tgt, enc_mask, dec_enc_mask, dec_mask)

    # 마지막 위치는 정답이 없으므로 [:-1]만 사용해 정답과 차원 맞춤
    loss = loss_function(gold, predictions[:, :-1])

    # 역전파로 gradient 계산
    loss.backward()

    # 최종적으로 optimizer.step()이 사용됩니다.
    optimizer.step()

    return loss, enc_attns, dec_attns, dec_enc_attns


In [ ]:
# ===== 학습 루프 =====

from tqdm import tqdm

BATCH_SIZE = 64   # 미니배치 크기
EPOCHS = 20       # 전체 에폭 수

# 매 에폭마다 번역 결과를 확인할 샘플 문장들
examples = [
    "오바마는 대통령이다.",
    "시민들은 도시 속에 산다.",
    "커피는 필요 없다.",
    "일곱 명의 사망자가 발생했다."
]

for epoch in range(EPOCHS):
    total_loss = 0

    # 배치 시작 인덱스 리스트 생성 후 셔플 (매 에폭 순서 변경)
    idx_list = list(range(0, enc_train.shape[0], BATCH_SIZE))
    random.shuffle(idx_list)
    t = tqdm(idx_list)  # 진행률 표시

    for (batch, idx) in enumerate(t):
        # 한 배치 학습 (numpy 슬라이싱 → torch.LongTensor 변환)
        # 수정: 옵티마이저가 model 파라미터로 생성되었으므로 model을 학습해야 함
        batch_loss, enc_attns, dec_attns, dec_enc_attns = \
        train_step(torch.LongTensor(enc_train[idx:idx+BATCH_SIZE]),
                   torch.LongTensor(dec_train[idx:idx+BATCH_SIZE]),
                   model,
                   optimizer)

        total_loss += batch_loss

        # 진행률 바에 현재 평균 손실 표시
        t.set_description('Epoch %2d' % (epoch + 1))
        t.set_postfix({'Loss': '%.4f' % (total_loss.item() / (batch + 1))})

    # 에폭 끝마다 샘플 문장 번역 결과 확인
    for example in examples:
        translate(example, model, ko_tokenizer, en_tokenizer)  # 수정: transformer → model

        # 기존 넘파이 슬라이싱 데이터를 torch.LongTensor로 감싸서 전달합니다.


Epoch  1:   3%|▎         | 33/1127 [04:10<2:19:38,  7.66s/it, Loss=11507.7888]

학습진행이 계속 멈춰서 진행을 할 수 없음(주피터->colab->kaggle 순서로 옮겨가면서 진행하였고 최대 epoch1에서 48프로까지 진행하다가 멈췄음)

In [ ]:
# ===== Attention 시각화 함수 =====
# 학습된 모델이 어디에 주목하는지 히트맵으로 그려봅니다.

def visualize_attention(src, tgt, enc_attns, dec_attns, dec_enc_attns):
    def draw(data, ax, x="auto", y="auto"):
        """하나의 attention head를 히트맵으로 그리는 헬퍼."""
        import seaborn
        seaborn.heatmap(data,
                        square=True,           # 정사각형 셀
                        vmin=0.0, vmax=1.0,    # 가중치 범위
                        cbar=False, ax=ax,
                        xticklabels=x,         # x축: 소스/타깃 토큰
                        yticklabels=y)         # y축: 소스/타깃 토큰

    # --- 인코더 self-attention 시각화 ---
    for layer in range(0, 2, 1):
        fig, axs = plt.subplots(1, 4, figsize=(20, 10))
        print("Encoder Layer", layer + 1)
        for h in range(4):  # 첫 4개 헤드만 그리기
            draw(enc_attns[layer][0, h, :len(src), :len(src)], axs[h], src, src)
        plt.show()

    # --- 디코더 self-attention + cross-attention 시각화 ---
    for layer in range(0, 2, 1):
        # (1) 디코더 self-attention
        fig, axs = plt.subplots(1, 4, figsize=(20, 10))
        print("Decoder Self Layer", layer+1)
        for h in range(4):
            draw(dec_attns[layer][0, h, :len(tgt), :len(tgt)], axs[h], tgt, tgt)
        plt.show()

        # (2) 인코더-디코더 cross-attention (타깃 토큰이 소스의 어디를 보는지)
        print("Decoder Src Layer", layer+1)
        fig, axs = plt.subplots(1, 4, figsize=(20, 10))
        for h in range(4):
            draw(dec_enc_attns[layer][0, h, :len(tgt), :len(src)], axs[h], src, tgt)
        plt.show()


In [ ]:
# ===== 번역(추론) 함수 =====
# Greedy decoding: 매 step마다 확률 최대인 토큰을 하나씩 생성

def evaluate(sentence, model, src_tokenizer, tgt_tokenizer):
    # 1) 입력 문장 전처리
    sentence = preprocess_sentence(sentence)

    # 2) 소스 문장을 토큰(piece)와 ID로 변환
    pieces = src_tokenizer.encode_as_pieces(sentence)
    tokens = src_tokenizer.encode_as_ids(sentence)

    # 3) 배치 차원 추가하여 인코더 입력 생성
    _input = torch.tensor(tokens).unsqueeze(0)

    ids = []  # 생성된 토큰 ID들
    # 디코더는 <bos>로 시작
    output = torch.tensor([tgt_tokenizer.bos_id()]).unsqueeze(0)

    # 최대 출력 길이만큼 반복하며 한 토큰씩 생성
    for i in range(dec_train.shape[-1]):
        # 현재까지의 입력에 대한 마스크 생성
        enc_padding_mask, combined_mask, dec_padding_mask = generate_masks(_input, output)

        # 모델 forward
        predictions, enc_attns, dec_attns, dec_enc_attns = \
        model(_input, output, enc_padding_mask, combined_mask, dec_padding_mask)

        # 마지막 위치의 분포에서 argmax (greedy 선택)
        predicted_id = torch.argmax(torch.softmax(predictions, dim=-1)[0, -1]).item()

        # <eos>가 나오면 종료
        if tgt_tokenizer.eos_id() == predicted_id:
            result = tgt_tokenizer.decode_ids(ids)
            return pieces, result, enc_attns, dec_attns, dec_enc_attns

        # 생성된 토큰을 결과에 추가하고 디코더 입력에 이어 붙임
        ids.append(predicted_id)
        output = torch.cat([output, torch.tensor([[predicted_id]])], dim=-1)

    # 최대 길이까지 갔다면 그대로 디코딩하여 반환
    result = tgt_tokenizer.decode_ids(ids)

    return pieces, result, enc_attns, dec_attns, dec_enc_attns


In [ ]:
# ===== 번역 + (선택적) Attention 시각화 통합 함수 =====

def translate(sentence, model, src_tokenizer, tgt_tokenizer, plot_attention=False):
    # 모델로부터 번역 결과 + attention 가중치들 받기
    pieces, result, enc_attns, dec_attns, dec_enc_attns = \
    evaluate(sentence, model, src_tokenizer, tgt_tokenizer)

    # 입력 문장과 예측 번역 출력
    print('Input: %s' % (sentence))
    print('Predicted translation: {}'.format(result))

    # 옵션이 켜져 있으면 attention 시각화도 같이 출력
    if plot_attention:
        visualize_attention(pieces, result.split(), enc_attns, dec_attns, dec_enc_attns)
